# film revenue prediction - project notebook

## sections
1. project overview
2. data acquisition
3. data merging
4. data cleaning
5. feature engineering
6. data leakage analysis
7. ablation study
8. fine-tuning experiment
9. interpretability with shap
10. evaluation metrics
11. recommended models + tuning
12. pipeline summary and timeline
13. report-ready outputs
14. pitfalls checklist

In [ ]:
# setup
import os
import json
import numpy as np
import pandas as pd

from pathlib import Path

# optional ml imports (install if needed)
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

## 1) project overview

### 1.1 central research question
- what matters more for pre-release prediction: feature representation or model choice?

### 1.2 prediction target
- primary target: `y = log1p(revenue)`
- derived label: `profitable = 1[revenue > 1.5 * budget]`

### 1.3 core thesis
- combining structured metadata + synopsis embeddings + poster embeddings should outperform any single modality.

## 2) data acquisition

### 2.1 tmdb (kaggle)
- load `tmdb_movies.csv`

### 2.2 imdb non-commercial files
- `title.crew.tsv.gz`
- `title.principals.tsv.gz`
- `name.basics.tsv.gz`

### 2.3 poster images
- build url: `https://image.tmdb.org/t/p/w342/{poster_path}`
- cache locally before embedding extraction

In [ ]:
# load raw datasets
# update paths to your actual files
TMDB_PATH = DATA_DIR / "tmdb_movies.csv"
CREW_PATH = DATA_DIR / "title.crew.tsv.gz"
PRINCIPALS_PATH = DATA_DIR / "title.principals.tsv.gz"
NAMES_PATH = DATA_DIR / "name.basics.tsv.gz"

tmdb = pd.read_csv(TMDB_PATH)
crew = pd.read_csv(CREW_PATH, sep='\t', low_memory=False)
principals = pd.read_csv(PRINCIPALS_PATH, sep='\t', low_memory=False)
names = pd.read_csv(NAMES_PATH, sep='\t', low_memory=False)

tmdb.head()

## 3) data merging

- use tmdb as base table
- join tmdb `imdb_id` to imdb `tconst`
- keep join diagnostics: row counts and match rate

In [ ]:
# merge scaffold
base = tmdb.copy()

# example: keep relevant imdb columns before merge
crew_small = crew[["tconst", "directors"]].copy()

merged = base.merge(
    crew_small,
    how="left",
    left_on="imdb_id",
    right_on="tconst"
)

match_rate = merged["tconst"].notna().mean()
print(f"rows: {len(base):,} -> {len(merged):,}")
print(f"match rate: {match_rate:.2%}")

## 4) data cleaning

- remove invalid/missing budget or revenue rows
- parse dates and normalize schema
- define train-ready base table

In [ ]:
# cleaning scaffold
df = merged.copy()

df = df[df["budget"].fillna(0) > 0]
df = df[df["revenue"].fillna(0) > 0]

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df.dropna(subset=["release_date"])

# targets
df["y_log_revenue"] = np.log1p(df["revenue"])
df["profitable"] = (df["revenue"] > 1.5 * df["budget"]).astype(int)

print(df.shape)
df[["budget", "revenue", "y_log_revenue", "profitable"]].head()

## 5) feature engineering

### 5.1 group 1: structured metadata
- runtime, language, release timing, production info

### 5.2 group 2: creative content
- genre indicators (handcrafted)
- synopsis embeddings (transfer learning)

### 5.3 group 3: talent features
- director/cast history with strict temporal cutoff
- poster embeddings (transfer learning, image modality)

In [ ]:
# feature blocks scaffold
feature_blocks = {
    "g1_structured": [],
    "g2_genre_handcrafted": [],
    "g2_synopsis_embeddings": [],
    "g3_talent_temporal": [],
    "g3_poster_embeddings": []
}

# placeholder: fill each block and then concatenate into design matrices
# X_g1 = ...
# X_g2 = ...
# X_g3 = ...

## 6) data leakage analysis

- define pre-release information boundary
- explicitly exclude post-release signals from main training (e.g., ratings/votes)
- run a separate leakage demonstration experiment

## 7) ablation study

- fixed split and evaluation protocol
- run experiment matrix (e0-e8)
- compare modality contributions and interactions

## 8) fine-tuning experiment

- fine-tune one transfer model variant and compare against frozen baseline

## 9) interpretability with shap

- explain best model globally and locally

## 10) evaluation metrics

- regression: rmse, mae, r2
- optional classification view for profitability: precision/recall/f1/auc

## 11) recommended models + hyperparameter tuning

- baseline linear/ridge
- tree ensembles (rf/xgboost/lightgbm/catboost if available)
- tuned best candidate

## 12) pipeline summary and timeline

- final pipeline diagram/table
- weekly execution plan

## 13) report-ready outputs

- export key figures/tables for final report

## 14) pitfalls checklist

- no leakage
- temporal consistency for talent features
- reproducible splits and seeds